In [492]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
import numpy as np
from datasets import load_dataset, concatenate_datasets

In [493]:
# Special Tokens

IGNORE_TOKEN = "<IGNORE>"
MASK_TOKEN = "<MASK>"
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
CLS_TOKEN = "<CLS>"
SEP_TOKEN = "<SEP>"

SPECIAL_TOKENS = [IGNORE_TOKEN, MASK_TOKEN, PAD_TOKEN, UNK_TOKEN, CLS_TOKEN, SEP_TOKEN]

# Exercise 8: Encoder

# Loading data

In [494]:
from gensim.downloader import load as gensim_load

glove = gensim_load("glove-wiki-gigaword-100")

In [495]:
full_ds_dict = load_dataset("imdb")
ds_train_full = full_ds_dict["train"]
ds_test_full = full_ds_dict["test"]

In [496]:
from datasets import Dataset as HFDataset


def sample_imdb(ds: HFDataset, n_samples: int) -> HFDataset:
    """
    Sample n_samples from the imdb dataset, ensuring that the dataset is balanced.
    """
    # How many samples per class
    n_per_class = n_samples // 2

    # Filter each class
    pos = (
        ds.filter(lambda x: x["label"] == 1).shuffle(seed=42).select(range(n_per_class))
    )
    neg = (
        ds.filter(lambda x: x["label"] == 0).shuffle(seed=42).select(range(n_per_class))
    )

    # Combine and shuffle
    balanced = concatenate_datasets([pos, neg]).shuffle(seed=42)

    return balanced


In [497]:
assert isinstance(ds_train_full, HFDataset)
ds_train = sample_imdb(ds_train_full, 1000)

assert (
    ds_train.filter(lambda x: x["label"] == 1).num_rows
    == ds_train.filter(lambda x: x["label"] == 0).num_rows
)

In [498]:
ds_train

Dataset({
    features: ['text', 'label'],
    num_rows: 1000
})

# Tokenization

In [499]:
all_count = 0
counts = {
    "<br />": 0,
    "<br/>": 0,
    "<br>": 0,
    "\n": 0,
}

for text in ds_train_full["text"]:  # type: ignore
    all_count += 1
    for br in counts:
        if br in text:
            counts[br] += 1

print(f"Number of reviews: {all_count}")
print()
for br, count in counts.items():
    print(
        f"Number of reviews with {br!r:>8}: {count:>6} ({count / all_count * 100:05.2f}%)"
    )

Number of reviews: 25000

Number of reviews with '<br />':  14665 (58.66%)
Number of reviews with  '<br/>':      0 (00.00%)
Number of reviews with   '<br>':      0 (00.00%)
Number of reviews with     '\n':      0 (00.00%)


In [500]:
for i, text in enumerate(ds_train_full["text"]):
    if "<br />" not in text:
        continue

    print(f"Review {i}")
    print(f"  {text}")
    sentences = text.split("<br />")
    empty_sentences = [sentence for sentence in sentences if not sentence.strip()]
    print(f"{len(empty_sentences)}/{len(sentences)} sentences are empty")

    for k, sentence in enumerate(sentences):
        print(f"  {k}: '{sentence}'")

    print()

    if i > 10:
        break

Review 0
  I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far b

In [501]:
def split_review(review: str) -> list[str]:
    return [sentence for sentence in review.split("<br />") if sentence.strip()]


split_review(
    "If only to avoid making this type of film in the future. This film is interesting as an experiment but tells no cogent story.<br /><br />One might feel virtuous for sitting thru it because it touches on so many IMPORTANT issues but it does so without any discernable motive. The viewer comes away with no new perspectives (unless one comes up with one while one's mind wanders, as it will invariably do during this pointless film).<br /><br />One might better spend one's time staring out a window at a tree growing.<br /><br />"
)

['If only to avoid making this type of film in the future. This film is interesting as an experiment but tells no cogent story.',
 "One might feel virtuous for sitting thru it because it touches on so many IMPORTANT issues but it does so without any discernable motive. The viewer comes away with no new perspectives (unless one comes up with one while one's mind wanders, as it will invariably do during this pointless film).",
 "One might better spend one's time staring out a window at a tree growing."]

In [502]:
import re


def tokenize(text: str) -> list[str]:
    # Convert to lowercase
    text = text.lower()

    # Remove all numbers entirely from the text
    text = re.sub(r"\d+", "", text)

    # on word boundaries and ignores punctuation
    tokens: list[str] = re.findall(r"\w+", text)

    return tokens


sample_sentence = "This is a sentence with an unknown word: supermuel"

tokenized = tokenize(sample_sentence)
tokenized[:15]

['this', 'is', 'a', 'sentence', 'with', 'an', 'unknown', 'word', 'supermuel']

In [503]:
from typing import Any


def replace_unknown_tokens(
    tokens: list[str],
) -> list[str]:
    return [
        token if (token in glove or token in SPECIAL_TOKENS) else UNK_TOKEN
        for token in tokens
    ]


replace_unknown_tokens(["ThiIUHIUHZER3s", "is", "a", "sentence", SEP_TOKEN, PAD_TOKEN])

['<UNK>', 'is', 'a', 'sentence', '<SEP>', '<PAD>']

In [504]:
from typing import Any


def pad_or_truncate(
    tokens: list[str],
    sequence_length: int,
) -> list[str]:
    truncated = tokens[:sequence_length]
    return truncated + [PAD_TOKEN] * (sequence_length - len(truncated))


padded = pad_or_truncate(tokenized, sequence_length=15)
print(padded)

['this', 'is', 'a', 'sentence', 'with', 'an', 'unknown', 'word', 'supermuel', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>']


In [505]:
def preprocess_review(
    review: str,
) -> list[list[str]]:
    splitted_review = split_review(review)
    tokenized_sentences = [tokenize(part) for part in splitted_review]
    without_unknowns = [
        replace_unknown_tokens(sentence) for sentence in tokenized_sentences
    ]
    return without_unknowns


sample_sentence = (
    "This is a IUHEIU2H787 sentence with<br /> an unknown<br /><br /> word: supermuel"
)

preprocess_review(sample_sentence)

[['this', 'is', 'a', '<UNK>', 'sentence', 'with'],
 ['an', 'unknown'],
 ['word', '<UNK>']]

## Tokenize IMDB

In [506]:
def pre_process_dataset(
    ds: HFDataset,
) -> tuple[HFDataset, list[str], dict[str, int], dict[int, str]]:
    ds = ds.map(lambda x: {"sentences_tokens": preprocess_review(x["text"])})
    print(f"before filtering: {ds.num_rows} reviews")

    # Filter out reviews that don't contain the SEP_TOKEN,
    # as they are not suitable for the Next Sentence Prediction task.
    ds = ds.filter(lambda x: len(x["sentences_tokens"]) > 1)

    print(f"after filtering: {ds.num_rows} reviews")

    vocab: list[str] = sorted(
        list(
            set(
                token
                for review in ds["sentences_tokens"]
                for sentence in review
                for token in sentence
            )
        )
    )

    for special_token in SPECIAL_TOKENS:
        if special_token not in vocab:
            vocab.insert(0, special_token)

    word_to_idx = {word: i for i, word in enumerate(vocab)}
    idx_to_word = {i: word for word, i in word_to_idx.items()}

    ds = ds.map(
        lambda x: {
            "sentences_tokens_ids": [
                [word_to_idx[token] for token in sentence]
                for sentence in x["sentences_tokens"]
            ]
        }
    )

    return ds, vocab, word_to_idx, idx_to_word


ds_train, vocab, word_to_idx, idx_to_word = pre_process_dataset(
    ds_train_full,  # type: ignore
)

assert all(len(sentence) > 1 for sentence in ds_train["sentences_tokens_ids"])

ds_train

Map: 100%|██████████| 25000/25000 [00:02<00:00, 9516.84 examples/s]


before filtering: 25000 reviews


Filter: 100%|██████████| 25000/25000 [00:02<00:00, 10357.12 examples/s]


after filtering: 14544 reviews


Map: 100%|██████████| 14544/14544 [00:02<00:00, 5180.20 examples/s]


Dataset({
    features: ['text', 'label', 'sentences_tokens', 'sentences_tokens_ids'],
    num_rows: 14544
})

In [507]:
IGNORE_TOKEN_ID = word_to_idx[IGNORE_TOKEN]
MASK_TOKEN_ID = word_to_idx[MASK_TOKEN]
PAD_TOKEN_ID = word_to_idx[PAD_TOKEN]
UNK_TOKEN_ID = word_to_idx[UNK_TOKEN]
CLS_TOKEN_ID = word_to_idx[CLS_TOKEN]
SEP_TOKEN_ID = word_to_idx[SEP_TOKEN]

SPECIAL_TOKEN_IDS = [word_to_idx[token] for token in SPECIAL_TOKENS]

for token in SPECIAL_TOKENS:
    print(f"{token:^10} -> {word_to_idx[token]}")


 <IGNORE>  -> 4
  <MASK>   -> 3
  <PAD>    -> 2
  <UNK>    -> 5
  <CLS>    -> 1
  <SEP>    -> 0


In [508]:
print(f"Vocabulary size: {len(word_to_idx)}")

Vocabulary size: 54483


# PreTraining Dataset

In [509]:
from typing import Tuple

allowed_random_indexes = [i for i in range(len(vocab)) if i not in SPECIAL_TOKEN_IDS]
allowed_random_range = min(allowed_random_indexes), max(allowed_random_indexes)
assert all(id < allowed_random_range[0] for id in SPECIAL_TOKEN_IDS)


def apply_15_80_10_10(input: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    device = input.device

    # randomly select 15% of tokens for the 80/10/10 Rule
    # Only select tokens that are not in SPECIAL_TOKEN_IDS
    is_not_special = ~torch.isin(input, torch.tensor(SPECIAL_TOKEN_IDS, device=device))
    selected = (torch.rand(input.shape, device=device) < 0.15) & is_not_special

    random = torch.rand(input.shape, device=device)
    # to_unch = selected & (random < 0.1) # These tokens won't be changed, so we ignore it.
    to_rand = (
        selected & (random >= 0.1) & (random < 0.2)
    )  # These tokens will be changed to random tokens
    to_mask = selected & (random >= 0.2)  # These tokens will be maked with MASK_TOKEN

    inputs = input.masked_fill(to_mask, MASK_TOKEN_ID)
    inputs[to_rand] = torch.randint(
        *allowed_random_range,
        size=(int(to_rand.sum()),),
        device=device,
    )

    return selected, inputs


torch.manual_seed(43)

test = torch.tensor([432, 1970, 2832, 1058, 590, 2630, 1970, 1911, 3175])
print(f"Input:    {test}")
selected, result = apply_15_80_10_10(test)
print(f"Masked:   {selected}")
print(f"Selected: {result}")

Input:    tensor([ 432, 1970, 2832, 1058,  590, 2630, 1970, 1911, 3175])
Masked:   tensor([False, False, False, False,  True,  True, False,  True, False])
Selected: tensor([ 432, 1970, 2832, 1058,    3,    3, 1970,    3, 3175])


In [510]:
import random


class PreTrainingDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset: HFDataset, sequence_length: int):
        self.hf_dataset = hf_dataset
        if "sentences_tokens_ids" not in self.hf_dataset.column_names:
            raise ValueError("sentences_tokens_ids column not found in dataset")
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        sample = self.hf_dataset[idx]

        sentences_tokens_ids: list[list[int]] = sample["sentences_tokens_ids"]
        sentences_count = len(sentences_tokens_ids)

        is_positive: bool = random.random() < 0.5
        k = random.randint(0, sentences_count - 2)
        if is_positive:
            sentence_1, sentence_2 = (
                sentences_tokens_ids[k],
                sentences_tokens_ids[k + 1],
            )
        else:
            sentence_1 = (
                sentences_tokens_ids[k]
                if random.random() < 0.5
                else sentences_tokens_ids[k + 1]
            )
            other_review = self.hf_dataset[random.randint(0, len(self.hf_dataset) - 1)][
                "sentences_tokens_ids"
            ]
            sentence_2 = random.choice(other_review)
        nsp_label = not is_positive

        # constuct the tensor
        input_tensor = torch.tensor(
            [CLS_TOKEN_ID]
            + sentence_1
            + [SEP_TOKEN_ID]
            + sentence_2
            + [SEP_TOKEN_ID]
            + [PAD_TOKEN_ID]
            * (self.sequence_length - len(sentence_1) - len(sentence_2) - 3),
            dtype=torch.long,
        )

        if len(input_tensor) > self.sequence_length:
            # TODO: what to do here ?
            # temporarily, we raise
            # raise ValueError(f"Input tensor length {len(input_tensor)} is greater than sequence length {self.sequence_length}")
            # okay, now we don't raise, but we truncate. BUT I REALLY DON'T LIKE THIS
            input_tensor = input_tensor[: self.sequence_length]

        # apply the 15/80/10/10 rule
        mask, target = apply_15_80_10_10(input_tensor)

        # we want to evaluate the model's predictions only on the mask.
        # for that, for every token non selected for masking, we'll put
        # the IGNORE token id at that position.
        # it will tell the loss function to ignore that position.
        mlm_label = input_tensor.masked_fill(~mask, IGNORE_TOKEN_ID)

        return target, mlm_label, nsp_label


# Create an instance of our custom dataset
pre_training_dataset_train = PreTrainingDataset(ds_train, sequence_length=512)

# Let's check one sample
target, mlm_label, nsp_label = pre_training_dataset_train[0]
print(f"Sample input tensor shape: {target.shape}")
print(f"Sample label tensor shape: {mlm_label.shape}")
print("-" * 20)

print(f"{target[:200]=}")
print(f"{mlm_label[:200]=}")
print(f"{nsp_label=}")

Sample input tensor shape: torch.Size([512])
Sample label tensor shape: torch.Size([512])
--------------------
target[:200]=tensor([    1, 23170,     3, 23170,  1351, 11112, 54070, 18712, 32144, 51984,
        46287,  3939, 33601,  1179, 48456, 10019, 48445, 47227, 25024, 53059,
        25024, 52653, 17629, 39689, 23669, 23170,  1290, 21681, 48445,     3,
            3, 25024, 52653, 42761,     3, 50286, 41481, 11176, 23290, 25024,
        16121, 49781, 49008, 15603,     3, 10425, 48522,     3,    12, 16861,
        33601, 17495,  9771, 10015, 23170, 39015, 20985, 49008, 42691, 48603,
        18174,     3,     0, 48456, 36430,     3,  7502,  2346,     3, 54177,
        47377,     3, 46567, 32280, 27569, 53179, 52573, 49008, 27407, 16143,
        43307,  6785,   138, 27840, 23669, 35017, 43307, 52573, 49008, 18051,
            3,  2787, 49008, 29127, 44936, 45051, 33601, 13620,     3,     3,
        48456,  3014, 47374, 48640,   138,  7537, 36613, 25020, 46839,  2460,
        48456, 520

# Multi-Head Self Attention

In [511]:
from math import sqrt


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, n_heads: int, d_model: int):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        self.d_model = d_model
        self.n_heads = n_heads

        self.d_k = d_model // n_heads
        assert self.d_k > 1

        # In normal self-attention, W_Q is of shape (d_model, d_k)
        # But here, we want n W_Q matrixes. Since we also want to
        # keep the computational needs similar to the original self-attention,
        # we'll use n **smaller** W_Q matrixes. A simple rule is to use
        # d_k = d_model // n_heads

        # But how to define n_heads matrixes ? Well, can we cheat and instead
        # create a bigger one that act as multiple smaller ones ?
        # We need n_heads matrixes of size (d_model, d_model // n_heads)
        # So this is exactly the same as one matrix of size (d_model, d_model)

        self.W_Q = nn.Linear(d_model, d_model)

        # Exactly the same for W_K
        self.W_K = nn.Linear(d_model, d_model)

        # In the normal self-attention, W_V is of size (d_model, d_v)
        # For simplicity, d_v is often set to d_k. So we can define
        # W_V of size (d_model, d_model) like the others
        self.W_V = nn.Linear(d_model, d_model)

        # final projection matrix
        self.W_O = nn.Linear(in_features=d_model, out_features=self.d_model)

    def forward(self, sentences: torch.Tensor, mask: torch.Tensor | None = None):
        # sentences is of shape (batch_size, sentence_length, d_model)
        batch_size, sentence_length, d_model = sentences.shape
        n_heads, d_k = self.n_heads, self.d_k

        assert d_model == self.d_model

        Q: torch.Tensor = self.W_Q(sentences)  # (batch_size, sentence_length, d_model)
        K: torch.Tensor = self.W_K(sentences)  # (batch_size, sentence_length, d_model)
        V: torch.Tensor = self.W_V(sentences)  # (batch_size, sentence_length, d_model)

        assert Q.shape == (batch_size, sentence_length, d_model)
        assert K.shape == (batch_size, sentence_length, d_model)
        assert V.shape == (batch_size, sentence_length, d_model)

        Q = Q.view(batch_size, sentence_length, n_heads, d_k)
        K = K.view(batch_size, sentence_length, n_heads, d_k)
        V = V.view(batch_size, sentence_length, n_heads, d_k)

        # Bring n_heads dimension up
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        assert Q.shape == (batch_size, n_heads, sentence_length, d_k)
        assert K.shape == (batch_size, n_heads, sentence_length, d_k)
        assert V.shape == (batch_size, n_heads, sentence_length, d_k)

        # Now we need to compute the raw scores by multiplying Q and K
        # The result should be of shape (batch_size, n_heads, sentence_length, sentence_length)
        # For that, we need to multiply something of shapes:
        # (batch_size, n_heads, sentence_length, d_k) (batch_size, n_heads, d_k, sentence_length)

        raw_scores = Q @ K.transpose(-1, -2)
        assert raw_scores.shape == (
            batch_size,
            n_heads,
            sentence_length,
            sentence_length,
        )

        # Now we scale
        scaled_scores = raw_scores / sqrt(d_k)

        # Masking
        if mask is not None:
            # The mask should already be broadcastable, e.g., (batch, 1, 1, seq_len)
            masked_scores = scaled_scores.masked_fill(mask, float("-inf"))
        else:
            masked_scores = scaled_scores

        # Compute the attention scores
        attention_scores = F.softmax(masked_scores, dim=-1)
        assert attention_scores.shape == (
            batch_size,
            n_heads,
            sentence_length,
            sentence_length,
        )  # Shape should not change

        # Get the values
        # Outputs should be of shape (batch_size, n_heads, sentence_length, d_k)
        outputs = attention_scores @ V
        # (batch_size, n_heads, sentence_length,sentence_length) @ (batch_size, n_heads, sentence_length, d_k)
        # = (batch_size, n_heads, sentence_length, d_k)
        assert outputs.shape == (batch_size, n_heads, sentence_length, d_k)

        # concatenate outputs into (batch_size, sentence_length, d_model)
        outputs = (
            outputs.transpose(1, 2)
            .contiguous()
            .view(batch_size, sentence_length, d_model)
        )

        # Apply last linear transformation
        return self.W_O(outputs)


# Encoder

### My ModuleList

In [512]:
from textwrap import dedent
from typing import Self


class MyModuleList(nn.Module):
    def __init__(self, modules: list[nn.Module]):
        super().__init__()

        for i, module in enumerate(modules):
            self.add_module(str(i), module)
        self._num_modules = len(modules)

    def __getitem__(self, idx: int) -> nn.Module:
        """Allows accessing modules by index, e.g., my_list[0]"""
        if not (-self._num_modules <= idx < self._num_modules):
            raise IndexError("Index out of range")
        if idx < 0:
            idx += self._num_modules

        return self._modules[str(idx)]

    def __len__(self) -> int:
        return self._num_modules

    def __repr__(self):
        modules = self._modules
        if not modules:
            return "MyModuleList()"

        s = "MyModuleList(\n"
        for i, module in modules.items():
            s += f"  ({i}): {module}\n"
        s += ")"

        return s

    def append(self, module: nn.Module) -> Self:
        if not isinstance(module, nn.Module):
            raise TypeError("Can only append nn.Module objects")

        self.add_module(str(self._num_modules), module)
        self._num_modules += 1
        return self


modules = [nn.Linear(4, 5), nn.Linear(3, 4)]
my_module_list = MyModuleList(modules)

print(my_module_list)

torch_module_list = nn.ModuleList(modules)
assert my_module_list.__repr__()[2:] == torch_module_list.__repr__()
assert len(my_module_list) == len(torch_module_list)
assert len(list(my_module_list.parameters())) == len(
    list(torch_module_list.parameters())
)

MyModuleList(
  (0): Linear(in_features=4, out_features=5, bias=True)
  (1): Linear(in_features=3, out_features=4, bias=True)
)


In [513]:
def create_embedding_matrix(word_to_idx: dict[str, int]) -> torch.Tensor:
    """
    Create an embedding matrix from a GloVe model and a word-to-index mapping.

    Args:
        glove: The GloVe model instance.
        word_to_idx: Mapping from word to index.

    Returns:
        A torch.Tensor of shape (vocab_size, embedding_dim).
    """
    embedding_dim: int = glove.vector_size  # type: ignore
    embedding_matrix = np.zeros((len(word_to_idx), embedding_dim))
    for word, i in word_to_idx.items():
        if word in glove:
            embedding_matrix[i] = glove[word]  # type: ignore
        # else, it remains a zero vector (for <unk>, <pad>, etc.)
    embedding_matrix_tensor = torch.tensor(embedding_matrix, dtype=torch.float32)
    return embedding_matrix_tensor


pretrained_embedding_matrix = create_embedding_matrix(word_to_idx)
print(f"{pretrained_embedding_matrix.shape=}")

pretrained_embedding_matrix.shape=torch.Size([54483, 100])


In [ ]:
from torch.nn.functional import relu


class PointWiseFeedForward(nn.Module):
    def __init__(self, d_model: int, inner_dim: int | None = None):
        super().__init__()
        if not inner_dim:
            # It's common to have a larger inner dimension
            inner_dim = d_model * 4
            self.inner_dim = inner_dim

        self.linear_1 = nn.Linear(d_model, self.inner_dim)
        self.linear_2 = nn.Linear(self.inner_dim, d_model)

    def forward(self, x):
        x = self.linear_1(x)
        x = relu(x)
        x = self.linear_2(x)
        return x


class EncoderBlock(nn.Module):
    def __init__(
        self, vocab_size: int, d_model: int, n_heads: int, dropout_p: float = 0.1
    ):
        super().__init__()
        self.n_heads = n_heads
        self.d_model = d_model

        self.multi_head_self_attention = MultiHeadSelfAttention(
            n_heads=n_heads, d_model=d_model
        )
        self.layer_norm1 = nn.LayerNorm(normalized_shape=self.d_model)
        self.ff = PointWiseFeedForward(d_model=d_model)
        self.layer_norm2 = nn.LayerNorm(normalized_shape=self.d_model)

        # Define dropout layers
        self.dropout1 = nn.Dropout(dropout_p)
        self.dropout2 = nn.Dropout(dropout_p)

    def forward(self, inputs: torch.Tensor, mask: torch.Tensor | None = None):
        batch_size, sentence_length, d_model = inputs.shape
        assert d_model == self.d_model

        attn_output = self.multi_head_self_attention(inputs, mask=mask)
        before_ffn = self.layer_norm1(inputs + self.dropout1(attn_output))
        ffn_output = self.ff(before_ffn)
        outputs = self.layer_norm2(before_ffn + self.dropout2(ffn_output))

        return outputs


class StackedEncoder(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        n_heads: int,
        sequence_length: int,
        depth: int = 6,
        embedding_matrix: torch.Tensor | None = None,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.depth = depth
        self.n_heads = n_heads
        self.sequence_length = sequence_length

        if embedding_matrix is not None:
            self.embeddings = nn.Embedding.from_pretrained(
                embedding_matrix, freeze=False
            )
        else:
            self.embeddings = nn.Embedding(
                num_embeddings=vocab_size, embedding_dim=d_model
            )

        self.pos_embeddings = nn.Embedding(
            num_embeddings=self.sequence_length, embedding_dim=d_model
        )

        self.blocks = MyModuleList(
            [
                EncoderBlock(
                    vocab_size=vocab_size,
                    n_heads=n_heads,
                    d_model=d_model,
                )
                for _ in range(depth)
            ]
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        # inputs.shape == (batch_size, sequence_length)
        device = inputs.device

        inputs = self.embeddings(inputs)  # (batch_size, sequence_length, d_model)
        positions = self.pos_embeddings(
            torch.arange(0, self.sequence_length, dtype=torch.long, device=device)
        )

        inputs = inputs + positions

        for layer in self.blocks:
            inputs = layer(inputs)
        return inputs

## Pretraining head

In [ ]:
class PreTrainingModel(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        n_heads: int,
        sequence_length: int,
        depth: int = 6,
        embedding_matrix: torch.Tensor | None = None,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.depth = depth
        self.n_heads = n_heads

        self.stacked_encoder = StackedEncoder(
            sequence_length=sequence_length,
            embedding_matrix=embedding_matrix,
            vocab_size=vocab_size,
            d_model=d_model,
            n_heads=n_heads,
            depth=depth,
        )
        self.mlm_head = nn.Linear(d_model, vocab_size)
        self.nsp_head = nn.Linear(d_model, 1)

    def forward(self, inputs: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # inputs.shape == (batch_size, sentence_length, d_model)
        outputs = self.stacked_encoder(inputs)
        # outputs.shape == (batch_size, sentence_length, d_model)

        mlm_logits = self.mlm_head(outputs)
        nsp_logit = self.nsp_head(outputs[:, 0, :])  # Only use the CLS token

        return mlm_logits, nsp_logit

# PreTraining

In [518]:
TRAIN_SAMPLES = 25000
SEQUENCE_LENGTH = 512

assert isinstance(ds_train_full, HFDataset)
ds_train_unpreprocessed = sample_imdb(ds_train_full, TRAIN_SAMPLES)

ds_train, vocab, word_to_idx, idx_to_word = pre_process_dataset(ds_train_unpreprocessed)

print(f"Samples: {len(ds_train)}")
print(f"Vocab size: {len(vocab)}")

pytorch_pretrain_dataset = PreTrainingDataset(ds_train, sequence_length=SEQUENCE_LENGTH)

input_tensor, mlm_label, nsp_label = pytorch_pretrain_dataset[0]
input_tensor[:50], mlm_label[:50], nsp_label

Map: 100%|██████████| 25000/25000 [00:03<00:00, 8328.73 examples/s]


before filtering: 25000 reviews


Filter: 100%|██████████| 25000/25000 [00:02<00:00, 10453.28 examples/s]


after filtering: 14544 reviews


Map: 100%|██████████| 14544/14544 [00:02<00:00, 5246.36 examples/s]

Samples: 14544
Vocab size: 54483


(tensor([    1,     3, 14490, 17136, 24669, 53062, 48456,  7222, 30329, 42709,
         49008, 17528, 12337,  1608, 23583, 23669, 48603, 31820,  1608, 23669,
         48472, 40973, 23170, 26207, 53612, 23290, 48546,     3, 42984, 33935,
         26308, 48546, 42709, 49008,  3837, 42984,  6438, 23170, 26207, 48582,
             3, 32090,  3837, 26308, 22496, 41481,  1290,    12,     3, 13081]),
 tensor([    4, 48456,     4,     4,     4,     4,     4,     4,     4,     4,
             4,     4,     4,     4,     4,     4,     4,     4,     4,     4,
             4,     4,     4,     4,     4,     4,     4, 52965,     4,     4,
             4,     4,     4,     4,     4,     4,     4,     4,     4,     4,
         48546,     4,     4,     4, 48518,     4,     4,     4, 35331,     4]),
 True)

### Actual training

In [ ]:
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import os


def train(
    model: nn.Module,
    dataset: PreTrainingDataset,
    device: str,
    epochs: int,
    batch_size: int,
    lr: float,
    save_path: str = "./models",
) -> None:
    model.to(device)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    mlm_loss_fn = nn.CrossEntropyLoss(ignore_index=IGNORE_TOKEN_ID)
    nsp_loss_fn = nn.BCEWithLogitsLoss()

    # Create directory for saving if it doesn't exist
    os.makedirs(save_path, exist_ok=True)

    for epoch in range(epochs):
        model.train()
        total_mlm_loss = 0.0
        total_nsp_loss = 0.0
        # Consider adding tqdm for a nice progress bar in Colab
        # from tqdm.notebook import tqdm
        # for batch in tqdm(dataloader, desc=f"Epoch {epoch + 1}/{epochs}"):
        for batch in dataloader:
            input_tensor, mlm_labels, nsp_labels = batch
            input_tensor = input_tensor.to(device)
            mlm_labels = mlm_labels.to(device)
            nsp_labels = nsp_labels.to(device).float()

            optimizer.zero_grad()
            mlm_logits, nsp_logits = model(input_tensor)

            mlm_loss = mlm_loss_fn(
                mlm_logits.view(-1, mlm_logits.size(-1)), mlm_labels.view(-1)
            )
            nsp_loss = nsp_loss_fn(nsp_logits.squeeze(-1), nsp_labels)
            loss = mlm_loss + nsp_loss
            loss.backward()
            optimizer.step()

            total_mlm_loss += mlm_loss.item()
            total_nsp_loss += nsp_loss.item()

        avg_mlm_loss = total_mlm_loss / len(dataloader)
        avg_nsp_loss = total_nsp_loss / len(dataloader)
        print(
            f"Epoch {epoch + 1}/{epochs} - "
            f"MLM Loss: {avg_mlm_loss:.4f} - NSP Loss: {avg_nsp_loss:.4f}"
        )

        # --- SAVE THE MODEL AT THE END OF EACH EPOCH ---
        # Save the full pre-training model
        torch.save(
            model.state_dict(),
            os.path.join(save_path, "imdb_pretrained_full_model_latest.pth"),
        )
        # Save just the encoder
        torch.save(
            model.stacked_encoder.state_dict(),
            os.path.join(save_path, "imdb_pretrained_encoder_latest.pth"),
        )

    print("Training finished. Final models saved.")

In [ ]:
# --- Model and Training Hyperparameters ---
D_MODEL = 100
N_HEADS = 5
DEPTH = 6
EPOCHS = 10  # Start with a moderate number of epochs
BATCH_SIZE = 16  # A safe starting point for Colab T4 GPUs
LR = 1e-4  # A smaller learning rate is often more stable for Transformers
SEQUENCE_LENGTH = 512
SAVE_PATH = "./pretrained_model"  # Directory to save model checkpoints

# --- Device Setup ---
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Using device: {DEVICE}")

# --- Initialize the Model ---
# Make sure 'vocab' and 'pretrained_embedding_matrix' are defined from your preprocessing steps
model = PreTrainingModel(
    vocab_size=len(vocab),
    d_model=D_MODEL,
    n_heads=N_HEADS,
    sequence_length=SEQUENCE_LENGTH,
    depth=DEPTH,
    embedding_matrix=pretrained_embedding_matrix,
)
print(
    f"Model initialized with {sum(p.numel() for p in model.parameters()):,} trainable parameters."
)


# --- Call the Training Function ---
# Ensure 'pytorch_pretrain_dataset' is created and ready
print("\nStarting pre-training...")
train(
    model=model,
    dataset=pytorch_pretrain_dataset,
    device=DEVICE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    save_path=SAVE_PATH,
)
print("Pre-training complete.")

# Down-stream task: Classification

In [532]:
import os
from torch.utils.data import DataLoader

class ClassificationDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset: HFDataset, word_to_idx: dict, sequence_length: int):
        self.hf_dataset = hf_dataset
        self.word_to_idx = word_to_idx
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        sample = self.hf_dataset[idx]
        review_text = sample['text']
        label = sample['label']

        tokenized = tokenize(review_text)
        without_unknowns = replace_unknown_tokens(tokenized)
        token_ids = [self.word_to_idx.get(token, UNK_TOKEN_ID) for token in without_unknowns]

        # 3. Truncate if necessary (leaving space for CLS and SEP)
        max_content_len = self.sequence_length - 2
        token_ids = token_ids[:max_content_len]

        # 4. Add special tokens
        input_ids = [CLS_TOKEN_ID] + token_ids + [SEP_TOKEN_ID]

        # 5. Pad to sequence_length
        padding_len = self.sequence_length - len(input_ids)
        input_ids = input_ids + ([PAD_TOKEN_ID] * padding_len)

        # 6. Convert to tensors
        input_tensor = torch.tensor(input_ids, dtype=torch.long)
        label_tensor = torch.tensor(label, dtype=torch.float32).unsqueeze(0)

        return input_tensor, label_tensor

In [533]:
class SentimentClassifier(nn.Module):
    """
    A sentiment classification model that uses a pre-trained StackedEncoder.
    """
    def __init__(self, vocab_size: int, d_model: int, n_heads: int, sequence_length: int, depth: int = 6):
        super().__init__()
        self.encoder = StackedEncoder(
            vocab_size=vocab_size,
            d_model=d_model,
            n_heads=n_heads,
            sequence_length=sequence_length,
            depth=depth,
        )
        self.dropout = nn.Dropout(0.1)
        self.classifier_head = nn.Linear(d_model, 1) # Output a single logit

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        # inputs.shape == (batch_size, sequence_length)
        encoder_output = self.encoder(inputs)
        # encoder_output.shape == (batch_size, sequence_length, d_model)

        cls_token_output = encoder_output[:, 0, :]
        # cls_token_output.shape == (batch_size, d_model)

        # Apply dropout and the final linear layer
        cls_token_output = self.dropout(cls_token_output)
        logits = self.classifier_head(cls_token_output)
        return logits

In [534]:
LR_FINETUNE = 2e-5  
EPOCHS_FINETUNE = 5 
BATCH_SIZE_FINETUNE = 32
# TRAIN_SAMPLES_CLF = 10000 
# TEST_SAMPLES_CLF = 2000

### Prepare datasets

In [535]:
# --- 1. Prepare Datasets and DataLoaders ---
print("Preparing classification datasets...")
# Use the full dataset and create balanced samples
ds_train_clf_unprocessed = sample_imdb(ds_train_full, 25000)
ds_test_clf_unprocessed = sample_imdb(ds_test_full, 2000)

# Create PyTorch dataset instances
train_clf_dataset = ClassificationDataset(ds_train_clf_unprocessed, word_to_idx, SEQUENCE_LENGTH)
test_clf_dataset = ClassificationDataset(ds_test_clf_unprocessed, word_to_idx, SEQUENCE_LENGTH)

# Create DataLoaders
train_clf_loader = DataLoader(train_clf_dataset, batch_size=BATCH_SIZE_FINETUNE, shuffle=True)
test_clf_loader = DataLoader(test_clf_dataset, batch_size=BATCH_SIZE_FINETUNE)
print(f"Created train dataloader with {len(train_clf_loader)} batches and test dataloader with {len(test_clf_loader)} batches.")

Preparing classification datasets...
Created train dataloader with 782 batches and test dataloader with 63 batches.


### Load pre-trained encoder weights

In [536]:
from pathlib import Path

classifier_model = SentimentClassifier(
    vocab_size=len(vocab),
    d_model=D_MODEL,
    n_heads=N_HEADS,
    sequence_length=SEQUENCE_LENGTH
).to(DEVICE)

# Load the pre-trained encoder weights
pretrained_encoder_path = Path("imdb_pretrained_encoder_latest.pth")
assert pretrained_encoder_path.exists()
print(f"Loading pre-trained weights from: {pretrained_encoder_path}")

encoder_state_dict = torch.load(pretrained_encoder_path, map_location=DEVICE)

classifier_model.encoder.load_state_dict(encoder_state_dict)
print("Pre-trained weights loaded successfully into the encoder!")

Loading pre-trained weights from: imdb_pretrained_encoder_latest.pth
Pre-trained weights loaded successfully into the encoder!


### Final training

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import copy

def evaluate_classifier(model: nn.Module, dataloader: DataLoader, criterion: nn.Module, device: str):
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            preds = torch.round(torch.sigmoid(outputs))
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    return avg_loss, accuracy, f1

def train_and_evaluate_classifier(
    model: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    device: str,
    epochs: int,
    lr: float,
):
    model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    best_model_wts = copy.deepcopy(model.state_dict())
    best_f1 = -1.0

    print("\nStarting fine-tuning...")
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_labels = []
        train_preds = []

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            preds = torch.round(torch.sigmoid(outputs))
            
            train_labels.extend(labels.cpu().detach().numpy())
            train_preds.extend(preds.cpu().detach().numpy())

        # Calculate training metrics for the epoch
        avg_train_loss = train_loss / len(train_loader)
        train_accuracy = accuracy_score(train_labels, train_preds)
        train_f1 = f1_score(train_labels, train_preds)

        # Evaluate on the test set
        test_loss, test_accuracy, test_f1 = evaluate_classifier(model, test_loader, criterion, device)

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.4f}, Train F1: {train_f1:.4f} | "
              f"Test Loss: {test_loss:.4f}, Test Acc: {test_accuracy:.4f}, Test F1: {test_f1:.4f}")

        # Save the best model based on test F1 score
        if test_f1 > best_f1:
            best_f1 = test_f1
            best_model_wts = copy.deepcopy(model.state_dict())
            print(f"  -> New best model saved with F1 score: {best_f1:.4f}")

    # Load best model weights
    model.load_state_dict(best_model_wts)
    return model

fine_tuned_model = train_and_evaluate_classifier(
    model=classifier_model,
    train_loader=train_clf_loader,
    test_loader=test_clf_loader,
    device=DEVICE,
    epochs=EPOCHS_FINETUNE,
    lr=LR_FINETUNE,
)


Starting fine-tuning...


KeyboardInterrupt: 